In [0]:
from pyspark.sql import functions as F

volume_path = "/Volumes/dataworkspace/default/data"

# 1. Ingest Raw Data
raw_telemetry = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{volume_path}/Iot telemetry.csv")
raw_events = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{volume_path}/Event.csv")

# 2. Validate Telemetry
valid_telemetry = (
    raw_telemetry
    .dropDuplicates(["timestamp", "asset_id", "sensor_id"])
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .filter(
        F.col("timestamp").isNotNull() &
        F.col("asset_id").isNotNull() &
        (F.trim(F.col("asset_id")) != "") &
        F.col("site_id").isNotNull() &
        F.col("building_id").isNotNull() &
        F.col("power_consumption (kW)").isNotNull() &
        F.col("temperature (°C)").isNotNull() &
        F.col("humidity (%)").isNotNull()
    )
)

# 3. Validate Events
valid_events = (
    raw_events
    .dropDuplicates(["event_id"])
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .filter(
        F.col("timestamp").isNotNull() & 
        F.col("asset_id").isNotNull() & 
        (F.trim(F.col("asset_id")) != "") &
        F.col("event_type").isNotNull()
    )
)

In [0]:
# 1. Hourly & Daily Transformations
hourly_curated = (
    valid_telemetry
    .withColumn("hour_window", F.date_trunc("hour", "timestamp"))
    .groupBy("site_id", "building_id", "asset_id", "hour_window")
    .agg(
        F.sum("power_consumption (kW)").alias("hourly_energy_consumption_kWh"),
        F.avg("temperature (°C)").alias("avg_temperature_C"),
        F.avg("humidity (%)").alias("avg_humidity_pct"),
        F.avg("pressure (hPa)").alias("avg_pressure_hPa"),
        F.avg("vibration (mm/s)").alias("avg_vibration")
    )
)

fault_statistics = (
    valid_events
    .filter(F.upper(F.col("event_type")).isin(["FAULT", "ERROR", "FAILURE"]))
    .groupBy("asset_id")
    .agg(
        F.count("event_id").alias("total_fault_count"),
        F.max("timestamp").alias("last_fault_timestamp")
    )
)

# 2. Aggregations (Asset, Building, Site)
asset_metrics = (
    hourly_curated
    .groupBy("asset_id")
    .agg(
        F.sum("hourly_energy_consumption_kWh").alias("total_energy_kWh"),
        F.avg("avg_temperature_C").alias("overall_avg_temp_C"),
        F.avg("avg_humidity_pct").alias("overall_avg_humidity_pct")
    )
    .join(fault_statistics, on="asset_id", how="left")
    .fillna(0, subset=["total_fault_count"])
)

building_metrics = (
    hourly_curated
    .groupBy("site_id", "building_id")
    .agg(
        F.sum("hourly_energy_consumption_kWh").alias("total_building_energy_kWh"),
        F.avg("avg_temperature_C").alias("building_avg_temp_C"),
        F.avg("avg_humidity_pct").alias("building_avg_humidity_pct"),
        F.countDistinct("asset_id").alias("total_assets")
    )
)

site_metrics = (
    hourly_curated
    .groupBy("site_id")
    .agg(
        F.sum("hourly_energy_consumption_kWh").alias("total_site_energy_kWh"),
        F.avg("avg_temperature_C").alias("site_avg_temp_C"),
        F.avg("avg_humidity_pct").alias("site_avg_humidity_pct"),
        F.countDistinct("building_id").alias("total_buildings"),
        F.countDistinct("asset_id").alias("total_assets")
    )
)

# 3. Save to Unity Catalog Tables
asset_metrics.write.mode("overwrite").saveAsTable("dataworkspace.default.asset_metrics")
building_metrics.write.mode("overwrite").saveAsTable("dataworkspace.default.building_metrics")
site_metrics.write.mode("overwrite").saveAsTable("dataworkspace.default.site_metrics")
print("Pipeline executed and tables updated successfully!")

Pipeline executed and tables updated successfully!
